# Huginn v2 - Servidor de Segmentación en Google Colab

Este notebook ejecuta el servicio de segmentación asíncrono para **Huginn v2** utilizando **TotalSegmentator 2.18.0**.

---

### Requisitos previos en Colab
1. **GPU activa**: Menú `Entorno de ejecución` -> `Cambiar tipo de entorno de ejecución` -> Seleccionar **GPU T4**.
2. **Secretos configurados** (ícono de llave en la barra lateral izquierda):
   - `NGROK_AUTHTOKEN` (Obligatorio): Tu authtoken de ngrok.
   - `HUGINN_API_TOKEN` (Recomendado): Token Bearer para autorizar las llamadas de Huginn.
   - `HUGINN_TOTALSEG_LICENSE` (Opcional): Tu clave académica para `tissue_4_types`. Si no la configuras, el servicio funcionará en modo degradado (solo detección anatómica de L3).
   - `NGROK_DOMAIN` (Opcional): Si dispones de un dominio estático o personalizado en ngrok.
   - `GITHUB_TOKEN` (Opcional): Solo si el repositorio es privado (Personal Access Token con permiso de lectura).

In [ ]:
#@title Celda 1: Parámetros y descarga del código del servicio
REPO_URL = "https://github.com/DC-CodeDev/huginn-colab.git" #@param {type:"string"}
RAMA_O_ETIQUETA = "main" #@param {type:"string"}
NGROK_DOMAIN = "" #@param {type:"string"}
APP_URL = "" #@param {type:"string"}

import os
import subprocess
import sys
from urllib.parse import urlparse, urlunparse

def _leer_secreto_colab(nombre: str) -> str:
    try:
        from google.colab import userdata
        val = userdata.get(nombre)
        return str(val).strip() if val else ""
    except Exception:
        return os.environ.get(nombre, "").strip()

repo_url = REPO_URL.strip()
ref = RAMA_O_ETIQUETA.strip() or "main"

if not repo_url:
    raise ValueError(
        "Falta el parámetro REPO_URL en la Celda 1. "
        "Completá la URL HTTPS del repositorio (por ejemplo: https://github.com/usuario/huginn-colab.git) "
        "y volvé a ejecutar."
    )

ngrok_authtoken = _leer_secreto_colab("NGROK_AUTHTOKEN")
if not ngrok_authtoken:
    raise ValueError(
        "Falta el Secret NGROK_AUTHTOKEN en Google Colab. "
        "Abrí el panel de Secretos (ícono de llave a la izquierda), creá NGROK_AUTHTOKEN con tu token de ngrok "
        "y habilitá el acceso para este notebook."
    )

if NGROK_DOMAIN.strip():
    os.environ["NGROK_DOMAIN"] = NGROK_DOMAIN.strip()
if APP_URL.strip():
    os.environ["HUGINN_APP_URL"] = APP_URL.strip().rstrip("/")

github_token = _leer_secreto_colab("GITHUB_TOKEN")
clone_url = repo_url
if github_token and repo_url.startswith("https://"):
    parsed = urlparse(repo_url)
    netloc = f"x-access-token:{github_token}@{parsed.hostname}"
    if parsed.port:
        netloc += f":{parsed.port}"
    clone_url = urlunparse((parsed.scheme, netloc, parsed.path, parsed.params, parsed.query, parsed.fragment))

def _sanitizar(texto: str) -> str:
    return texto.replace(github_token, "***") if github_token else texto

base_dir = "/content" if os.path.isdir("/content") else os.getcwd()
target_dir = os.path.join(base_dir, "colab")

if os.path.isdir(os.path.join(target_dir, ".git")):
    print(f"Actualizando repositorio existente en {target_dir} (ref: {ref})...")
    res = subprocess.run(
        ["git", "-C", target_dir, "fetch", "--depth", "1", "origin", ref],
        capture_output=True,
        text=True,
    )
    if res.returncode != 0:
        raise RuntimeError(f"Error al actualizar el repositorio: {_sanitizar(res.stderr.strip())}")
    res_co = subprocess.run(
        ["git", "-C", target_dir, "checkout", "-q", "FETCH_HEAD"],
        capture_output=True,
        text=True,
    )
    if res_co.returncode != 0:
        raise RuntimeError(f"Error al cambiar a {ref}: {_sanitizar(res_co.stderr.strip())}")
else:
    print(f"Clonando repositorio en {target_dir} (ref: {ref})...")
    res = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", ref, clone_url, target_dir],
        capture_output=True,
        text=True,
    )
    if res.returncode != 0:
        raise RuntimeError(
            "Error al clonar el repositorio. Verificá REPO_URL, RAMA_O_ETIQUETA "
            "y el Secret GITHUB_TOKEN si el repo es privado. "
            f"Detalle: {_sanitizar(res.stderr.strip())}"
        )

# Soportar tanto repo independiente (raíz = paquete colab) como monorepo (subcarpeta colab/)
if os.path.exists(os.path.join(target_dir, "colab", "__init__.py")):
    import_root = target_dir
    pkg_dir = os.path.join(target_dir, "colab")
else:
    import_root = base_dir
    pkg_dir = target_dir

if import_root not in sys.path:
    sys.path.insert(0, import_root)
os.environ["HUGINN_COLAB_DIR"] = pkg_dir

import colab
print(f"Código del servicio listo en: {pkg_dir}")
print(colab.format_version_banner())
_stale_msg = colab.check_code_staleness(pkg_dir)
if _stale_msg:
    print(f"\n⚠️ AVISO: {_stale_msg}\n")

In [ ]:
# Celda 2: Verificar versión de Python y aceleración por GPU
!python --version
!nvidia-smi

In [ ]:
#@title Celda 3: Instalación robusta e idempotente de dependencias
import os
from colab.install_deps import install_requirements

pkg_dir = os.environ.get("HUGINN_COLAB_DIR", "colab")
req_path = os.path.join(pkg_dir, "requirements.txt")
install_requirements(req_path, timeout_seconds=300)

### Diagnóstico opcional del entorno
La celda siguiente es **opcional** y no es necesaria para iniciar el servicio. Ejecutala únicamente si necesitás reportar las versiones exactas de Python, CUDA, `torch`, `numpy` y `pip freeze` en Colab para ajustar `requirements.txt`.

In [ ]:
#@title [OPCIONAL] Celda de diagnóstico: versiones del entorno Colab
import sys
import subprocess

print("Python:", sys.version.replace("\n", " "))
for mod_name in ("torch", "numpy"):
    try:
        mod = __import__(mod_name)
        extra = f" (CUDA {mod.version.cuda})" if mod_name == "torch" else ""
        print(f"{mod_name}: {mod.__version__}{extra}")
    except Exception as exc:
        print(f"{mod_name}: no disponible ({exc})")

print("\n--- pip freeze ---")
res = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True)
print(res.stdout)

In [ ]:
# Celda 4: Activar licencia académica de TotalSegmentator si está en Secretos
import subprocess

try:
    from google.colab import userdata
    license_key = userdata.get("HUGINN_TOTALSEG_LICENSE")
    if license_key:
        res = subprocess.run(
            ["totalseg_set_license", "-l", license_key],
            capture_output=True,
            text=True,
            check=False,
        )
        if res.returncode == 0:
            print("Licencia académica configurada exitosamente.")
        else:
            print(f"totalseg_set_license falló (código: {res.returncode}).")
    else:
        print("HUGINN_TOTALSEG_LICENSE no configurada en Secretos. Modo degradado (solo L3).")
except Exception:
    print("No se pudo ejecutar totalseg_set_license.")

In [ ]:
# Celda 5: Precalentamiento de modelos (Warmup)
# Descarga previa de pesos para evitar latencias en la primera solicitud del usuario
import os
import numpy as np
import nibabel as nib
from totalsegmentator.python_api import totalsegmentator

print("Precalentando modelos...")
dummy_data = np.zeros((32, 32, 16), dtype=np.int16)
dummy_affine = np.eye(4)
dummy_img = nib.Nifti1Image(dummy_data, dummy_affine)
dummy_path = "/tmp/warmup_dummy.nii.gz"
dummy_out = "/tmp/warmup_out"
os.makedirs(dummy_out, exist_ok=True)
nib.save(dummy_img, dummy_path)

try:
    # Precalentar modelo de vértebras
    totalsegmentator(dummy_path, dummy_out, task="total", roi_subset=["vertebrae_L3"], device="gpu", fast=True)
    print("Modelo de vértebras precalentado.")
except Exception as e:
    print("Warmup rápido de vértebras completado o advertencia:", e)

print("Precalentamiento finalizado.")

### Dominio fijo en ngrok (opcional) y enlace de conexión
- Para usar siempre la misma URL, consultá tu dominio gratuito en **panel de ngrok > Domains** (`https://dashboard.ngrok.com/domains`) y pegalo en `NGROK_DOMAIN` (en la Celda 1 o en Secretos).
- Si no definís `NGROK_DOMAIN`, ngrok asignará una URL distinta en cada sesión.
- Si definís `APP_URL` en la Celda 1, al finalizar se imprimirá un enlace clicable `<APP_URL>/#/conectar/<código>` además del código `HGN1-...`.
- La Celda 6 queda en bucle monitoreando los logs del servicio. Para volver a ejecutarla, primero detenela con el botón **Stop** (detener la celda corta solo el monitor y **no apaga el servicio**).

In [ ]:
#@title Celda 6: Iniciar servicio Huginn y túnel ngrok
import os
from colab.server import start_server_colab

url = start_server_colab(
    ngrok_domain=os.environ.get("NGROK_DOMAIN"),
    app_url=os.environ.get("HUGINN_APP_URL"),
)